<a href="https://colab.research.google.com/github/samuel-verburg/ASSA2026_Sound_field_decomposition/blob/main/Exercises/exercise2_wave_expansions/exercise2_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### ASSA 2026 Autumn School: Sound Field Decomposition Using Transducer Arrays
**Exercise 2: Wave expansions and regularisation**

**Introduction**

In lecture 3 we wrote the pressure measured by an array as a linear combination of elementary waves, $\mathbf{p}=\mathbf{A}\mathbf{x}+\mathbf{n}$, and saw that estimating the wave amplitudes $\mathbf{x}$ requires regularisation. In this notebook we use the same model for finding where the sources are.

We expand the sound field at the array into point sources (monopoles) placed on a grid of candidate positions. The estimated amplitudes $\hat{\mathbf{x}}$ form a map of the sources, which depends to a large extent on the regularisation we choose.

In this notebook you will:
- Simulate the pressure measured by a planar array in front of a few sources and build an equivalent source expansion.
- Formulate the localisation as a regularised linear optimisation problem and solve it with $\ell_2$ and $\ell_1$ penalties.
- Choose the regularisation parameter, and see what happens when the sources do not fit the expansion.

Each task is described first and followed by a code cell to complete: replace the `...` placeholders.

The notebook needs `numpy`, `matplotlib` and [`cvxpy`](https://www.cvxpy.org/index.html), all available in Colab. In MATLAB, the equivalent of `cvxpy` is [`cvx`](https://cvxr.com/cvx/doc/install.html).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp

**Equivalent source expansion**

Consider sound sources on the plane $z=0$ and an array of $M$ microphones on the plane $z=d$. We describe the sound field at the array as the field radiated by $N$ monopoles placed on a grid of candidate positions $\mathbf{r}_n$ on the source plane,

$$p(\mathbf{r}_m)=\sum_{n=1}^{N}x_n\,G(\mathbf{r}_m,\mathbf{r}_n)+n_m,\qquad G(\mathbf{r}_m,\mathbf{r}_n)=\frac{e^{-\mathrm{i}kR_{mn}}}{4\pi R_{mn}}, \tag{1}$$

where $R_{mn}=|\mathbf{r}_m-\mathbf{r}_n|$, $k=\omega/c$, $x_n$ is the strength of the $n$-th monopole, and $n_m$ is noise. In matrix form,

$$\mathbf{p}=\mathbf{A}\mathbf{x}+\mathbf{n},\qquad A_{mn}=G(\mathbf{r}_m,\mathbf{r}_n). \tag{2}$$

Each column of $\mathbf{A}$ is the pressure that one candidate source would produce at the microphones. If there are only a few sources the true $\mathbf{x}$ is zero everywhere except at source positions. Localising the sources then amounts to estimating $\mathbf{x}$ from $\mathbf{p}$ and finding where $|\hat{x}_n|$ is largest.

We provide some functions to help with the plotting. `plot_setup` shows a top view of the array, the candidate grid and the true sources. `plot_grid_map` plots any quantity defined on the candidate grid, and `plot_source_map` plots $|\hat{x}_n|$ in dB re its maximum.

In [ ]:
def plot_setup(ax, r_mic, r_grid, r_src):
    # Top view of the set-up: microphones, candidate grid and true sources.
    ax.plot(r_grid[:, 0], r_grid[:, 1], '.', color='0.75', ms=3, label='candidate grid')
    ax.plot(r_mic[:, 0], r_mic[:, 1], 'ko', ms=5, mfc='none', label='microphones')
    ax.plot(r_src[:, 0], r_src[:, 1], 'r+', ms=10, mew=2, label='true sources')
    ax.set_aspect('equal')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.legend(loc='upper right', fontsize=8)


def plot_grid_map(ax, values, r_grid, r_src=None, title='', clabel='', vmin=None, vmax=None):
    # Plot values defined on the square candidate grid.
    g = np.unique(r_grid[:, 0])
    im = ax.pcolormesh(g, g, values.reshape(g.size, g.size), shading='nearest', cmap='viridis', vmin=vmin, vmax=vmax)
    if r_src is not None:
        ms = 10 if len(r_src) < 5 else 3
        ax.plot(r_src[:, 0], r_src[:, 1], 'r+', ms=ms, mew=1.5)
    ax.set_aspect('equal')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, label=clabel)


def plot_source_map(ax, x, r_grid, r_src=None, title='', dyn_range=20):
    # Plot |x| in dB re its maximum.
    LdB = 20 * np.log10(np.abs(x) / np.abs(x).max() + 1e-12)
    plot_grid_map(ax, LdB, r_grid, r_src, title, 'dB re max', vmin=-dyn_range, vmax=0)

**Set-up**

The candidate sources form a $21\times21$ grid with 5 cm spacing, covering $1\times1\ \mathrm{m}^2$ of the plane $z=0$ ($N=441$). The array has $M=32$ microphones at random positions over a $0.6\times0.6\ \mathrm{m}^2$ aperture, at $d=0.5$ m from the source plane. Two monopoles with different strengths radiate at 1 kHz. Note that the problem has many more unknowns than equations ($N\gg M$).

In [ ]:
c = 343.0                       # speed of sound (m/s)
f = 1000.0                      # frequency (Hz)
snr = 20                        # signal-to-noise ratio (dB)
rng = np.random.default_rng(0)  # random number generator (fixed seed)

# Candidate source positions: 21 x 21 grid with 5 cm spacing on the plane z = 0
g = np.linspace(-0.5, 0.5, 21)
GX, GY = np.meshgrid(g, g)
r_grid = np.column_stack([GX.ravel(), GY.ravel(), np.zeros(GX.size)])
N = r_grid.shape[0]

# Array: M microphones at random positions over a 0.6 x 0.6 m aperture on the plane z = d
M, aperture, d = 32, 0.6, 0.5
r_mic = np.column_stack([rng.uniform(-aperture / 2, aperture / 2, (M, 2)), np.full(M, d)])

# True sources: two monopoles on the grid with different strengths and phases
r_src = np.array([[-0.1, 0.0, 0.0],
                  [ 0.1, 0.1, 0.0]])
q_src = np.array([1.0, 0.7 * np.exp(1j * np.pi / 4)])

**ex2.A**:

Simulate the measurements.
- Complete `distance`, which returns the matrix of distances between two sets of points, and `greens`, which evaluates the Green's function in Eq. (1) between them. Both take arrays of shape `(P, 3)` and `(Q, 3)` and return a `(P, Q)` matrix.
- Complete `simulate_measurement`: compute the pressure at the microphones radiated by sources at positions `r_s` with strengths `q_s`, and add complex Gaussian noise scaled to the given signal-to-noise ratio, $\|\mathbf{n}\|_2/\|\mathbf{p}_0\|_2=10^{-\mathrm{SNR}/20}$. You can draw $M$ samples from a standard normal distribution with `rng.standard_normal(M)`. Remember that the noise must be complex.

Return the noise as well. We will need it later.

In [ ]:
def distance(r, r0):
    # Distances between the points r (P x 3) and r0 (Q x 3), as a P x Q matrix.
    return ...  # TO DO


def greens(r, r0, f):
    # Free-field Green's function e^{-ikR} / (4 pi R) between r (P x 3) and r0 (Q x 3).
    ...  # TO DO
    return ...  # TO DO


def simulate_measurement(r_s, q_s, f, snr=snr):
    # Noisy pressure at the microphones due to monopoles at r_s with strengths q_s.
    p0 = ...  # TO DO
    noise = ...  # TO DO
    return p0 + noise, noise


p, noise = simulate_measurement(r_src, q_src, f)

fig, ax = plt.subplots(figsize=(5, 4))
plot_setup(ax, r_mic, r_grid, r_src)
ax.set_title(f'Top view, array at z = {d} m')
plt.show()

**ex2.B**:

Build the expansion matrix $\mathbf{A}$ of Eq. (2) using `greens`.

Before solving anything, check how similar the columns of $\mathbf{A}$ are. Complete `coherence_map`, which computes the coherence between column $i$ and every column of $\mathbf{A}$,

$$\beta_{in}=\frac{|\mathbf{a}_i^\mathrm{H}\mathbf{a}_n|}{\|\mathbf{a}_i\|_2\|\mathbf{a}_n\|_2}.$$

Plot the coherence with the column of the first source, at 500 Hz, 1 kHz and 2 kHz.

Questions: How large is the region where $\beta$ is close to one, and how does it change with frequency? At 1 kHz, what is the coherence between the columns of the two sources?

In [ ]:
A = ...  # TO DO
print(f'A is {A.shape[0]} x {A.shape[1]}')


def coherence_map(A, i):
    # Coherence between column i of A and every column of A.
    ...  # TO DO
    return ...  # TO DO


i_src = np.argmin(distance(r_grid, r_src), axis=0)   # grid indices of the true sources
print(f'Coherence between the two sources at {f:.0f} Hz: {coherence_map(A, i_src[0])[i_src[1]]:.2f}')

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, fi in zip(axs, [500.0, 1000.0, 2000.0]):
    beta = coherence_map(greens(r_mic, r_grid, fi), i_src[0])
    plot_grid_map(ax, beta, r_grid, r_src, f'Coherence with source 1, {fi:.0f} Hz', r'$\beta$', vmin=0, vmax=1)
plt.tight_layout()
plt.show()

*Observations.* TO DO...

**Optimisation problem**

Since $N>M$, infinitely many $\mathbf{x}$ produce the measured pressure, thus we need to specify which one we want. We do so with the cost function that we minimise. We will compare three choices (lecture 3):

- Minimum norm (no regularisation, the data are fitted exactly):
$$\hat{\mathbf{x}}_\mathrm{MN}=\arg\min_{\mathbf{x}}\|\mathbf{x}\|_2\quad\text{subject to}\quad\mathbf{A}\mathbf{x}=\mathbf{p}.$$
- $\ell_2$ (Tikhonov) regularisation, which favours smooth solutions with low energy:
$$\hat{\mathbf{x}}_{\ell_2}=\arg\min_{\mathbf{x}}\|\mathbf{A}\mathbf{x}-\mathbf{p}\|_2^2+\lambda^2\|\mathbf{x}\|_2^2.$$
- $\ell_1$ regularisation, which favours solutions with few non-zero entries (sparse):
$$\hat{\mathbf{x}}_{\ell_1}=\arg\min_{\mathbf{x}}\tfrac12\|\mathbf{A}\mathbf{x}-\mathbf{p}\|_2^2+\lambda\|\mathbf{x}\|_1,\qquad\|\mathbf{x}\|_1=\sum_n|x_n|.$$

All three problems are convex. Instead of writing a solver for each of them, we describe the cost function and the constraints with `cvxpy`, and let the package find the minimiser. For example, the minimum-norm solution is obtained with

```python
x = cp.Variable(N, complex=True)                
cost = cp.norm(x, 2)                            
constraints = [A @ x == p]                      
prob = cp.Problem(cp.Minimize(cost), constraints)
prob.solve()
x_mn = x.value
```

Changing the regulariser only changes the line that defines `cost`. Useful functions are `cp.sum_squares(v)` for $\|\mathbf{v}\|_2^2$ and `cp.norm(v, 1)` for $\|\mathbf{v}\|_1$.

**ex2.C**:

Complete the functions `solve_l2` and `solve_l1`, and compute the three solutions. Plot the source maps $|\hat{x}_n|$ and print the estimated strengths at the two source positions.

Start with `lam_l2 = 0.5` and `lam_l1 = 0.05`, and then try values one or two orders of magnitude larger and smaller.

Questions: Which solutions locate both sources? How do the estimated strengths compare with the true ones ($|q_1|=1$, $|q_2|=0.7$)? Why does the $\ell_2$ solution look like the coherence map of ex2.B?

In [ ]:
def solve_mn(A, p):
    x = cp.Variable(A.shape[1], complex=True)
    cost = cp.norm(x, 2)
    prob = cp.Problem(cp.Minimize(cost), [A @ x == p])
    prob.solve()
    return x.value


def solve_l2(A, p, lam):
    ...  # TO DO
    return ...  # TO DO


def solve_l1(A, p, lam):
    ...  # TO DO
    return ...  # TO DO


lam_l2 = 0.5
lam_l1 = 0.05

x_mn = solve_mn(A, p)
x_l2 = solve_l2(A, p, lam_l2)
x_l1 = solve_l1(A, p, lam_l1)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, xh, name in zip(axs, [x_mn, x_l2, x_l1], ['Minimum norm', r'$\ell_2$', r'$\ell_1$']):
    plot_source_map(ax, xh, r_grid, r_src, name)
plt.tight_layout()
plt.show()

for xh, name in zip([x_mn, x_l2, x_l1], ['MN', 'l2', 'l1']):
    print(f'{name:3s}: |x| at sources = {np.abs(xh[i_src]).round(2)}, max |x| = {np.abs(xh).max():.2f}')

*Observations.* TO DO...

**Choosing the regularisation parameter**

So far we chose $\lambda$ by hand, knowing where the sources were. In a real measurement we don't have that luxury. The regularisation problems above can also be written in constrained form, where the parameter $\lambda$ is replaced by a bound $\varepsilon$ on the misfit:

$$\hat{\mathbf{x}}_{\ell_2}=\arg\min_{\mathbf{x}}\|\mathbf{x}\|_2\quad\text{s.t.}\quad\|\mathbf{A}\mathbf{x}-\mathbf{p}\|_2\leq\varepsilon,\qquad
\hat{\mathbf{x}}_{\ell_1}=\arg\min_{\mathbf{x}}\|\mathbf{x}\|_1\quad\text{s.t.}\quad\|\mathbf{A}\mathbf{x}-\mathbf{p}\|_2\leq\varepsilon.$$

Choosing $\varepsilon\approx\|\mathbf{n}\|_2$ means that we don't try to fit the data more closely than the noise level (discrepancy principle). The advantage is that $\varepsilon$ has a physical meaning, and can be estimated, e.g., from a background noise measurement.

**ex2.D**:

- Solve the $\ell_1$ problem for `lam = 0.002, 0.02, 0.2, 0.5`. For each solution, plot the source map, and compute the number of non-zero coefficients (e.g., entries above 1% of the maximum) and the misfit relative to the noise, $\|\mathbf{A}\hat{\mathbf{x}}-\mathbf{p}\|_2/\|\mathbf{n}\|_2$.
- Complete `solve_l2_constrained` and `solve_l1_constrained` using `cvxpy` constraints, and solve both with $\varepsilon=\|\mathbf{n}\|_2$.

Questions: What happens to the map and the misfit when $\lambda$ is too small or too large? Which $\lambda$ gives a misfit similar to the noise? Do the constrained solutions agree with your choices in ex2.C?

In [ ]:
lams = [0.002, 0.02, 0.2, 0.5]
fig, axs = plt.subplots(1, len(lams), figsize=(20, 4))
for ax, lam in zip(axs, lams):
    xh = ...  # TO DO
    nnz = ...  # TO DO
    misfit = ...  # TO DO
    plot_source_map(ax, xh, r_grid, r_src, rf'$\lambda$ = {lam}: {nnz} non-zeros, misfit/noise = {misfit:.2f}')
plt.tight_layout()
plt.show()


def solve_l2_constrained(A, p, eps):
    ...  # TO DO
    return ...  # TO DO


def solve_l1_constrained(A, p, eps):
    ...  # TO DO
    return ...  # TO DO


eps = np.linalg.norm(noise)
x_l2_eps = solve_l2_constrained(A, p, eps)
x_l1_eps = solve_l1_constrained(A, p, eps)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
plot_source_map(axs[0], x_l2_eps, r_grid, r_src, r'$\ell_2$, $\varepsilon=\|\mathbf{n}\|_2$')
plot_source_map(axs[1], x_l1_eps, r_grid, r_src, r'$\ell_1$, $\varepsilon=\|\mathbf{n}\|_2$')
plt.tight_layout()
plt.show()

*Observations.* TO DO...

**ex2.E**:

So far the sources were exactly on the grid, i.e., the true field could be described by two columns of $\mathbf{A}$. This is rarely the case in practice. Move the second source to $(0.125, 0.075, 0)$ m, halfway between grid points, and solve the $\ell_2$ and $\ell_1$ problems (constrained form) at 1 kHz and 2 kHz.

Questions: How does the $\ell_1$ solution represent a source that is not on the grid? Would a finer grid solve the problem? What would happen to the coherence between neighbouring columns?

In [ ]:
r_src_off = np.array([[-0.1, 0.0, 0.0],
                      [0.125, 0.075, 0.0]])

fig, axs = plt.subplots(2, 2, figsize=(10, 8))
for row, fi in enumerate([1000.0, 2000.0]):
    A_f = ...  # TO DO
    p_f, noise_f = ...  # TO DO
    x_l2_off = ...  # TO DO
    x_l1_off = ...  # TO DO
    plot_source_map(axs[row, 0], x_l2_off, r_grid, r_src_off, rf'$\ell_2$, off-grid source, {fi:.0f} Hz')
    plot_source_map(axs[row, 1], x_l1_off, r_grid, r_src_off, rf'$\ell_1$, off-grid source, {fi:.0f} Hz')
    print(f'{fi:.0f} Hz, l1: largest |x| = {np.sort(np.abs(x_l1_off))[::-1][:5].round(2)}')
plt.tight_layout()
plt.show()

*Observations.* TO DO...

**Bonus exercises**

- *Debiasing.* The $\ell_1$ penalty shrinks the estimated strengths. Once the support (non-zero entries) is found, re-estimate the strengths by least squares using only those columns of $\mathbf{A}$. Are the strengths now closer to the true values?
- *Elastic net.* Combine the $\ell_1$ and $\ell_2$ penalties.
- *Array geometry.* Replace the random array by a compact array (e.g., $0.2\times0.2\ \mathrm{m}^2$) or by a regular $6\times6$ grid of microphones. How do the coherence maps and the source maps change?
- *Sound field reconstruction.* Once $\hat{\mathbf{x}}$ is known, Eq. (1) predicts the pressure anywhere in front of the source plane. Remove some microphones from the problem, predict their pressure with each solution, and use the prediction error to choose $\lambda$.

**References**
- [Hansen1998] P. C. Hansen, *Rank-Deficient and Discrete Ill-Posed Problems: Numerical Aspects of Linear Inversion*, SIAM, 1998.
- [Candes2008] E. J. Candès and M. B. Wakin, "An introduction to compressive sampling", *IEEE Signal Process. Mag.* 25(2), 21–30, 2008.
- [Xenaki2014] A. Xenaki, P. Gerstoft and K. Mosegaard, "Compressive beamforming", *J. Acoust. Soc. Am.* 136(1), 260–271, 2014.
- [FernandezGrande2017] E. Fernandez-Grande, A. Xenaki and P. Gerstoft, "A sparse equivalent source method for near-field acoustic holography", *J. Acoust. Soc. Am.* 141(1), 532–542, 2017.